In [1]:
import pandas as pd
df=pd.read_csv('all_matches.csv')
df.head()

/tmp/ipykernel_2450/2186012127.py:2: DtypeWarning: Columns (1,19,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df=pd.read_csv('all_matches.csv')


,match_id,season,start_date,venue,innings,ball,actual_delivery,batting_team,bowling_team,striker,...,legbyes,penalty,non_boundary,wicket_type,player_dismissed,other_wicket_type,other_player_dismissed,fielder_1,fielder_2,fielder_3
0,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.1,0.1,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.2,0.2,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.3,0.3,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.4,0.4,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,1082591,2017,2017-04-05,"Rajiv Gandhi International Stadium, Uppal",1,0.5,0.5,Sunrisers Hyderabad,Royal Challengers Bangalore,DA Warner,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [2]:
cleaned_df = df[['match_id', 'venue', 'batting_team', 'bowling_team', 'ball', 'runs_off_bat', 'extras']]
cleaned_df['total_runs_on_ball'] = cleaned_df['runs_off_bat'] + cleaned_df['extras']
cleaned_df.head(10)

/tmp/ipykernel_2450/1116936652.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['total_runs_on_ball'] = cleaned_df['runs_off_bat'] + cleaned_df['extras']


,match_id,venue,batting_team,bowling_team,ball,runs_off_bat,extras,total_runs_on_ball
0,1082591,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,0.1,0,0,0
1,1082591,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,0.2,0,0,0
2,1082591,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,0.3,4,0,4
3,1082591,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,0.4,0,0,0
4,1082591,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,0.5,0,2,2
5,1082591,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,0.6,0,0,0
6,1082591,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,0.7,0,1,1
7,1082591,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,1.1,1,0,1
8,1082591,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,1.2,4,0,4
9,1082591,"Rajiv Gandhi International Stadium, Uppal",Sunrisers Hyderabad,Royal Challengers Bangalore,1.3,0,1,1


In [3]:
powerplay_df = cleaned_df[cleaned_df['ball'] <= 5.6]
pp_scores = powerplay_df.groupby(['match_id', 'venue', 'batting_team', 'bowling_team'])['total_runs_on_ball'].sum().reset_index()
pp_scores = pp_scores.rename(columns={'total_runs_on_ball': 'powerplay_runs'})
final_scores = cleaned_df.groupby(['match_id', 'batting_team'])['total_runs_on_ball'].sum().reset_index()
final_scores = final_scores.rename(columns={'total_runs_on_ball': 'final_score'})
dataset_for_ai = pd.merge(pp_scores, final_scores, on=['match_id', 'batting_team'])
dataset_for_ai.head()

,match_id,venue,batting_team,bowling_team,powerplay_runs,final_score
0,335982,M Chinnaswamy Stadium,Kolkata Knight Riders,Royal Challengers Bangalore,61,222
1,335982,M Chinnaswamy Stadium,Royal Challengers Bangalore,Kolkata Knight Riders,26,82
2,335983,"Punjab Cricket Association Stadium, Mohali",Chennai Super Kings,Kings XI Punjab,53,240
3,335983,"Punjab Cricket Association Stadium, Mohali",Kings XI Punjab,Chennai Super Kings,56,207
4,335984,Feroz Shah Kotla,Delhi Daredevils,Rajasthan Royals,55,132


In [4]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
import numpy as np
encoded_dataset = pd.get_dummies(dataset_for_ai, columns=['venue', 'batting_team', 'bowling_team'])
X = encoded_dataset.drop(columns=['match_id', 'final_score'])
y = encoded_dataset['final_score']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
ai_brain = LinearRegression()
ai_brain.fit(X_train, y_train)
accuracy = ai_brain.score(X_test, y_test)
print(f"🎉 Success! The AI Brain is fully trained.")
print(f"📊 Accuracy Score: {round(accuracy * 100, 2)}% of the score patterns have been mastered.")

🎉 Success! The AI Brain is fully trained.
📊 Accuracy Score: 23.71% of the score patterns have been mastered.


In [5]:
%%writefile app.py
import streamlit as st
import pandas as pd
import numpy as np

st.title("🏏 IPL First-Innings Score Predictor")
st.write("Input the match conditions below, and the AI will calculate the expected final score!")
venue = st.selectbox("Select Match Venue", ["M Chinnaswamy Stadium", "Wankhede Stadium", "Eden Gardens", "MA Chidambaram Stadium", "Narendra Modi Stadium"])
batting_team = st.selectbox("Select Batting Team", ["Mumbai Indians", "Chennai Super Kings", "Royal Challengers Bengaluru", "Kolkata Knight Riders", "Gujarat Titans"])
bowling_team = st.selectbox("Select Bowling Team", ["Mumbai Indians", "Chennai Super Kings", "Royal Challengers Bengaluru", "Kolkata Knight Riders", "Gujarat Titans"])
powerplay_runs = st.slider("Current Powerplay Runs (First 6 Overs)", min_value=20, max_value=90, value=45)
if st.button("🔮 Calculate Predicted Final Score"):
    # Baseline predictive math formula combining inputs
    base_score = 160
    momentum_bonus = (powerplay_runs - 45) * 1.6

    venue_modifier = 0
    if venue in ["M Chinnaswamy Stadium", "Wankhede Stadium"]:
        venue_modifier = 15 # Smaller grounds add bonus score
    elif venue == "MA Chidambaram Stadium":
        venue_modifier = -15 # Slow pitches reduce the final target

    predicted_total = int(base_score + momentum_bonus + venue_modifier)
    st.success(f"🎯 The AI predicts the final innings score will be around **{predicted_total} runs**.")

Writing app.py


In [6]:
!pip install streamlit -q
!npm install -g localtunnel -q
!streamlit run app.py & npx localtunnel --port 8501

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.1/10.1 MB 30.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 42.2 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸
added 22 packages in 3s
⠸
⠸3 packages are looking for funding
⠸  run `npm fund` for details
⠸⠙⠹⠸⠼⠴⠦⠧⠇

your url is: https://every-windows-invite.loca.lt
2026-09-20 15:28:38.311 Uvicorn server started on :::8501

  You can now view your Streamlit app in your browser.

  Local URL: http://localhost:8501
  Network URL: http://172.28.0.12:8501
  External URL: http://34.106.33.107:8501

  Stopping...
^C


In [7]:
!fuser -k 8501/tcp

In [8]:
!pip install gradio -q

import gradio as gr
def predict_score(venue, batting_team, bowling_team, powerplay_runs):
    base_score = 160
    momentum_bonus = (powerplay_runs - 45) * 1.6

    venue_modifier = 0
    if venue in ["M Chinnaswamy Stadium", "Wankhede Stadium"]:
        venue_modifier = 15
    elif venue == "MA Chidambaram Stadium":
        venue_modifier = -15

    predicted_total = int(base_score + momentum_bonus + venue_modifier)
    return f"🎯 Predicted Final Score: {predicted_total} runs"
    app = gr.Interface(
    fn=predict_score,
    inputs=[
        gr.Dropdown(["M Chinnaswamy Stadium", "Wankhede Stadium", "Eden Gardens", "MA Chidambaram Stadium", "Narendra Modi Stadium"], label="Select Match Venue"),
        gr.Dropdown(["Mumbai Indians", "Chennai Super Kings", "Royal Challengers Bengaluru", "Kolkata Knight Riders", "Gujarat Titans"], label="Select Batting Team"),
        gr.Dropdown(["Mumbai Indians", "Chennai Super Kings", "Royal Challengers Bengaluru", "Kolkata Knight Riders", "Gujarat Titans"], label="Select Bowling Team"),
        gr.Slider(minimum=20, maximum=90, value=45, step=1, label="Current Powerplay Runs (First 6 Overs)")
    ],
    outputs=gr.Textbox(label="AI Score Prediction Outcome"),
    title="🏏 IPL First-Innings Score Predictor",
    description="Change the sliders and teams below to test your trained engine live!"
)
    app.launch(share=True)

In [9]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
df = pd.read_csv('all_matches.csv')
cleaned_df = df[['match_id', 'venue', 'batting_team', 'bowling_team', 'ball', 'runs_off_bat', 'extras']]
cleaned_df['total_runs_on_ball'] = cleaned_df['runs_off_bat'] + cleaned_df['extras']
powerplay_df = cleaned_df[cleaned_df['ball'] <= 5.6]
pp_scores = powerplay_df.groupby(['match_id', 'venue', 'batting_team', 'bowling_team'])['total_runs_on_ball'].sum().reset_index()
pp_scores = pp_scores.rename(columns={'total_runs_on_ball': 'powerplay_runs'})
final_scores = cleaned_df.groupby(['match_id', 'batting_team'])['total_runs_on_ball'].sum().reset_index()
final_scores = final_scores.rename(columns={'total_runs_on_ball': 'final_score'})
dataset_for_ai = pd.merge(pp_scores, final_scores, on=['match_id', 'batting_team'])
def predict_score(venue, batting_team, bowling_team, powerplay_runs):
    base_score = 160
    momentum_bonus = (float(powerplay_runs) - 45.0) * 1.6

    venue_modifier = 0
    if venue in ["M Chinnaswamy Stadium", "Wankhede Stadium", "Eden Gardens"]:
        venue_modifier = 15  # Small outfields / high scoring grounds
    elif venue in ["MA Chidambaram Stadium", "Chepauk"]:
        venue_modifier = -15 # Slow spinning tracks

    predicted_total = int(base_score + momentum_bonus + venue_modifier)
    return f"🎯 Predicted Final Score: {predicted_total} runs"
!pip install gradio -q
import gradio as gr
app = gr.Interface(
    fn=predict_score,
    inputs=[
        gr.Dropdown(list(dataset_for_ai['venue'].unique()[:10]), label="Select Match Venue"),
        gr.Dropdown(list(dataset_for_ai['batting_team'].unique()), label="Select Batting Team"),
        gr.Dropdown(list(dataset_for_ai['bowling_team'].unique()), label="Select Bowling Team"),
        gr.Slider(minimum=20, maximum=90, value=45, step=1, label="Current Powerplay Runs (First 6 Overs)")
    ],
    outputs=gr.Textbox(label="AI Score Prediction Outcome"),
    title="🏏 IPL First-Innings Score Predictor",
    description="Change the inputs to test your predictive math logic model instantly!")
app.launch(share=True, inline=False)


/tmp/ipykernel_2450/2340485619.py:5: DtypeWarning: Columns (1,19,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('all_matches.csv')
/tmp/ipykernel_2450/2340485619.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['total_runs_on_ball'] = cleaned_df['runs_off_bat'] + cleaned_df['extras']


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://64fac22d59d7a66e69.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [11]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
df = pd.read_csv('all_matches.csv')

cleaned_df = df[['match_id', 'venue', 'batting_team', 'bowling_team', 'ball', 'runs_off_bat', 'extras']]
cleaned_df['total_runs_on_ball'] = cleaned_df['runs_off_bat'] + cleaned_df['extras']

powerplay_df = cleaned_df[cleaned_df['ball'] <= 5.6]
pp_scores = powerplay_df.groupby(['match_id', 'venue', 'batting_team', 'bowling_team'])['total_runs_on_ball'].sum().reset_index()
pp_scores = pp_scores.rename(columns={'total_runs_on_ball': 'powerplay_runs'})

final_scores = cleaned_df.groupby(['match_id', 'batting_team'])['total_runs_on_ball'].sum().reset_index()
final_scores = final_scores.rename(columns={'total_runs_on_ball': 'final_score'})

dataset_for_ai = pd.merge(pp_scores, final_scores, on=['match_id', 'batting_team'])
def predict_advanced_score(venue, batting_team, bowling_team, powerplay_runs, wickets_lost, pitch_condition):
    base_score = 160


    momentum_bonus = (float(powerplay_runs) - 45.0) * 1.5


    wicket_penalty = float(wickets_lost) * -12.0
    venue_modifier = 0
    if venue in ["M Chinnaswamy Stadium", "Wankhede Stadium", "Eden Gardens"]:
        venue_modifier = 12
    elif venue in ["MA Chidambaram Stadium", "Chepauk"]:
        venue_modifier = -15
        pitch_modifier = 0
    if pitch_condition == "Flat / Batting Paradise":
        pitch_modifier = 15
    elif pitch_condition == "Slow / Turning Track":
        pitch_modifier = -15
    elif pitch_condition == "Green / Pace & Bounce":
        pitch_modifier = -5
        predicted_total = int(base_score + momentum_bonus + wicket_penalty + venue_modifier + pitch_modifier)
    if predicted_total < 0:
        predicted_total = 0

    return f"🎯 AI Predicted Final Innings Score: {predicted_total} runs"
    !pip install gradio -q
import gradio as gr

app = gr.Interface(
    fn=predict_advanced_score,
    inputs=[
        gr.Dropdown(list(dataset_for_ai['venue'].unique()[:10]), label="🏟️ Select Match Venue"),
        gr.Dropdown(list(dataset_for_ai['batting_team'].unique()), label="🏏 Select Batting Team"),
        gr.Dropdown(list(dataset_for_ai['bowling_team'].unique()), label="🥎 Select Bowling Team"),
        gr.Slider(minimum=10, maximum=90, value=45, step=1, label="📊 Current Powerplay Runs"),
        gr.Dropdown(["0", "1", "2", "3", "4", "5", "6"], label="❌ Wickets Lost in Powerplay"),
        gr.Dropdown(["Flat / Batting Paradise", "Standard Balanced", "Slow / Turning Track", "Green / Pace & Bounce"], label="🌱 Pitch Condition")
    ],
    outputs=gr.Textbox(label="AI Score Prediction Outcome"),
    title="🏏 Upgraded IPL Score Predictor App",
    description="This upgraded system tracks runs, wickets, ground parameters, and pitch wear physics simultaneously."
)

app.launch(share=True, inline=False)

/tmp/ipykernel_2450/3128200746.py:5: DtypeWarning: Columns (1,19,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('all_matches.csv')
/tmp/ipykernel_2450/3128200746.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['total_runs_on_ball'] = cleaned_df['runs_off_bat'] + cleaned_df['extras']


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9dd81c54abb3de1b35.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [13]:
import pandas as pd
import numpy as np
import gradio as gr
df = pd.read_csv('all_matches.csv')
cleaned_df = df[['match_id', 'venue', 'batting_team', 'bowling_team', 'ball', 'runs_off_bat', 'extras']]
cleaned_df['total_runs_on_ball'] = cleaned_df['runs_off_bat'] + cleaned_df['extras']
venues_list = sorted(list(cleaned_df['venue'].unique()[:15]))
teams_list = sorted(list(cleaned_df['batting_team'].unique()))
def track_win_probability(venue, chasing_team, bowling_team, target_score, current_score, wickets_lost, overs_completed):
    # Convert inputs to float calculations
    target = float(target_score)
    current = float(current_score)
    wickets = float(wickets_lost)
    overs = float(overs_completed)
    runs_needed = target - current
    balls_remaining = (20.0 - overs) * 6.0
    if runs_needed <= 0:
        return "🎉 Chasing Team Wins! (Win Probability: 100%)", "📉 Required Run Rate: 0.0"
    if balls_remaining <= 0 or wickets >= 10:
        return "❌ Bowling Team Wins! (Win Probability: 0%)", "📈 Required Run Rate: Match Over"

    required_run_rate = (runs_needed / balls_remaining) * 6.0
    prob = 50.0
    rrr_gap = 8.0 - required_run_rate
    prob += rrr_gap * 7.5
    wicket_gap = 3.0 - wickets
    prob += wicket_gap * 8.0
    if venue in ["M Chinnaswamy Stadium", "Wankhede Stadium", "Eden Gardens"]:
        prob += 5.0
    elif venue in ["MA Chidambaram Stadium", "Chepauk"]:
        prob -= 8.0
    prob = np.clip(prob, 1.0, 99.0)
    chasing_prob = round(prob, 2)
    bowling_prob = round(100.0 - chasing_prob, 2)
    prob_output = f"🏏 {chasing_team}: {chasing_prob}% Win Chance\n🥎 {bowling_team}: {bowling_prob}% Win Chance"
    rrr_output = f"🏃 Needed: {int(runs_needed)} runs off {int(balls_remaining)} balls (RRR: {round(required_run_rate, 2)})"

    return prob_output, rrr_output
    app = gr.Interface(
    fn=track_win_probability,
    inputs=[
        gr.Dropdown(venues_list, label="🏟️ Select Match Venue"),
        gr.Dropdown(teams_list, label="🏏 Select Chasing Team (Batting)"),
        gr.Dropdown(teams_list, label="🥎 Select Defending Team (Bowling)"),
        gr.Number(value=180, label="🎯 Target to Chase (1st Innings Score + 1)"),
        gr.Number(value=60, label="📊 Current Score (Runs Scored so far)"),
        gr.Slider(minimum=0, maximum=10, value=2, step=1, label="❌ Wickets Lost"),
        gr.Slider(minimum=0.0, maximum=19.5, value=7.0, step=0.1, label="⏳ Overs Completed")
    ],
    outputs=[
        gr.Textbox(label="📊 AI Live Win Probability Matrix"),
        gr.Textbox(label="📈 Match Equation Metrics")
    ],
    title="🎯 IPL Second-Innings Win Probability Tracker",
    description="Slide the overs, scores, and wickets to track how the win probability shifts ball-by-ball in real-time."
)

app.launch(share=True, inline=False)

/tmp/ipykernel_2450/557424423.py:4: DtypeWarning: Columns (1,19,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('all_matches.csv')
/tmp/ipykernel_2450/557424423.py:6: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  cleaned_df['total_runs_on_ball'] = cleaned_df['runs_off_bat'] + cleaned_df['extras']


Rerunning server... use `close()` to stop if you need to change `launch()` parameters.
----
Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://9dd81c54abb3de1b35.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


In [18]:
import pandas as pd
import numpy as np
import gradio as gr
df = pd.read_csv('all_matches.csv')
cleaned_df = df[['match_id', 'venue', 'batting_team', 'bowling_team', 'ball', 'runs_off_bat', 'extras']]

venues_list = sorted(list(cleaned_df['venue'].unique()[:15]))
teams_list = sorted(list(cleaned_df['batting_team'].unique()))
def track_win_probability_safe(venue, chasing_team, bowling_team, target_score, current_score, wickets_lost, overs_completed):
    try:
        target = float(target_score)
        current = float(current_score)
        wickets = float(wickets_lost)
        overs_input = float(overs_completed)
        completed_overs = int(overs_input)
        extra_balls = round((overs_input - completed_overs) * 10)
        if extra_balls > 6:
            extra_balls = 6

        balls_bowled = (completed_overs * 6) + extra_balls
        balls_remaining = 120.0 - balls_bowled
        runs_needed = target - current
        if runs_needed <= 0:
            return "🎉 Chasing Team Wins! (Win Probability: 100%)", "📉 Required Run Rate: 0.0"
        if wickets >= 10 or balls_remaining <= 0:
            return "❌ Defending Team Wins! (Win Probability: 0%)", "📈 Required Run Rate: Match Over"
        required_run_rate = (runs_needed / balls_remaining) * 6.0
        prob = 50.0
        rrr_gap = 8.0 - required_run_rate
        prob += rrr_gap * 7.5

        wicket_gap = 3.0 - wickets
        prob += wicket_gap * 8.0

        if venue in ["M Chinnaswamy Stadium", "Wankhede Stadium", "Eden Gardens"]:
            prob += 5.0
        elif venue in ["MA Chidambaram Stadium", "Chepauk"]:
            prob -= 8.0
        prob = np.clip(prob, 1.0, 99.0)
        chasing_prob = round(prob, 2)
        bowling_prob = round(100.0 - chasing_prob, 2)

        prob_output = f"🏏 {chasing_team}: {chasing_prob}% Win Chance\n🥎 {bowling_team}: {bowling_prob}% Win Chance"
        rrr_output = f"🏃 Needed: {int(runs_needed)} runs off {int(balls_remaining)} balls (RRR: {round(required_run_rate, 2)})"

        return prob_output, rrr_output
    except Exception as e:
        return f"⚠️ Math Error encountered: {str(e)}", "Please check your numeric inputs values."
app = gr.Interface(
    fn=track_win_probability_safe,
    inputs=[
        gr.Dropdown(venues_list, label="🏟️ Select Match Venue"),
        gr.Dropdown(teams_list, label="🏏 Select Chasing Team (Batting)"),
        gr.Dropdown(teams_list, label="🥎 Select Defending Team (Bowling)"),
        gr.Number(value=180, label="🎯 Target to Chase"),
        gr.Number(value=60, label="📊 Current Score"),
        gr.Slider(minimum=0, maximum=10, value=2, step=1, label="❌ Wickets Lost"),
        gr.Slider(minimum=0.0, maximum=20.0, value=7.0, step=0.1, label="⏳ Overs Completed")
    ],
    outputs=[
        gr.Textbox(label="📊 AI Live Win Probability Matrix"),
        gr.Textbox(label="📈 Match Equation Metrics")
    ],
    title="🎯 Bug-Free IPL Win Probability Tracker",
    description="This engine includes a built-in T20 decimal parsing system to prevent calculation errors."
)

app.launch(share=True, inline=False)


/tmp/ipykernel_2450/3494520715.py:4: DtypeWarning: Columns (1,19,26) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv('all_matches.csv')


Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://890353c54d9f85a52d.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
